# Airbnb em Buenos Aires: o que explica o preço?

Continuação da análise exploratória. A pergunta agora é de quem anuncia: **dá para estimar o preço certo de um anúncio, e o que pesa nele?**

Método:
- Mesma limpeza da EDA: preço acima de zero, sem outliers de preço (regra do IQR) e no máximo 30 noites mínimas.
- Preço em log (a distribuição é assimétrica). Distância até o Obelisco calculada a partir de latitude/longitude.
- Comparação honesta com uma **regra simples** (mediana por bairro e tipo de imóvel), em validação cruzada de 5 folds.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

d = pd.read_csv("listings.csv")
d = d[d.price.notna() & (d.price > 0)].copy()
q1, q3 = d.price.quantile([.25, .75]); lim = q3 + 1.5*(q3 - q1)
d = d[(d.price <= lim) & (d.minimum_nights <= 30)].copy()
lat0, lon0 = -34.6037, -58.3816  # Obelisco
d["dist_km"] = np.hypot((d.latitude-lat0)*111, (d.longitude-lon0)*111*np.cos(np.radians(lat0)))
d["lp"] = np.log(d.price)
print(f"Anúncios após limpeza: {len(d):,} | preço mediano: ARS {d.price.median():,.0f}")

Anúncios após limpeza: 30,760 | preço mediano: ARS 29,264


In [2]:
hosts = d.groupby("host_id").calculated_host_listings_count.first()
pro = d.calculated_host_listings_count >= 10
print(f"Anfitriões com 10+ anúncios: {(hosts>=10).mean():.1%} dos anfitriões, mas {pro.mean():.1%} dos anúncios")
print(d.groupby("room_type").price.median().round(0).to_string())

Anfitriões com 10+ anúncios: 2.6% dos anfitriões, mas 29.5% dos anúncios
room_type
Entire home/apt    30694.0
Hotel room         11382.0
Private room       17300.0
Shared room        10926.0


In [3]:
X = pd.get_dummies(d[["neighbourhood", "room_type"]], drop_first=True).astype(float)
for c in ["dist_km", "minimum_nights", "number_of_reviews", "availability_365",
          "calculated_host_listings_count", "reviews_per_month"]:
    X[c] = d[c].fillna(0)
y, P = d.lp.values, d.price.values
kf = KFold(5, shuffle=True, random_state=42)

regra = np.zeros(len(d))
for i_tr, i_te in kf.split(d):
    g = d.iloc[i_tr].groupby(["neighbourhood", "room_type"]).lp.median(); gm = d.iloc[i_tr].lp.median()
    regra[i_te] = [g.get(k, gm) for k in zip(d.iloc[i_te].neighbourhood, d.iloc[i_te].room_type)]

previsoes = {
    "Regra: mediana bairro + tipo": regra,
    "Regressão linear": cross_val_predict(LinearRegression(), X, y, cv=kf),
    "Gradient boosting": cross_val_predict(HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=0), X, y, cv=kf),
}
for nome, p in previsoes.items():
    print(f"{nome:30s} R² (log) {r2_score(y, p):.3f} | erro absoluto médio ARS {mean_absolute_error(P, np.exp(p)):,.0f}")

Regra: mediana bairro + tipo   R² (log) 0.177 | erro absoluto médio ARS 10,178
Regressão linear               R² (log) 0.214 | erro absoluto médio ARS 10,046
Gradient boosting              R² (log) 0.308 | erro absoluto médio ARS 9,361


In [4]:
lin = LinearRegression().fit(X, y); coef = pd.Series(lin.coef_, X.columns)
print(f"Cada km a mais do Obelisco: {np.exp(coef['dist_km'])-1:+.1%} no preço")
print(((np.exp(coef.filter(like="room_type")) - 1)).map("{:+.0%}".format).to_string())
r = P / np.exp(previsoes["Gradient boosting"])
print(f"Anúncios 30%+ abaixo do preço estimado: {(r<0.7).mean():.0%} | 30%+ acima: {(r>1.3).mean():.0%}")

Cada km a mais do Obelisco: -1.0% no preço
room_type_Hotel room      -54%
room_type_Private room    -41%
room_type_Shared room     -58%
Anúncios 30%+ abaixo do preço estimado: 15% | 30%+ acima: 22%


## Leitura

- **Mercado concentrado:** menos de 3% dos anfitriões controlam quase 30% dos anúncios — a oferta já é, em boa parte, profissional.
- **Tipo de imóvel pesa mais que bairro:** quarto privativo sai ~40% mais barato que o imóvel inteiro, mantendo o resto constante; cada km de distância do centro tira ~1%.
- **O preço é difícil de prever só com o anúncio:** o melhor modelo explica ~31% da variação (em log) e reduz o erro em ~8% frente à regra simples. Fotos, comodidades e avaliação textual — que não estão nesta base — devem explicar boa parte do resto.
- **Uso prático:** cerca de 15% dos anúncios estão 30%+ abaixo do preço estimado — candidatos a revisão de preço para quem gerencia vários imóveis.